In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append(os.path.abspath('..'))

import time
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = "HuggingFaceTB/SmolLM2-135M"

# Download and load SmolLM2
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
W_list = []
layer_names = []

for n, m in model.named_modules():
    if isinstance(m, nn.Linear) and "lm_head" not in n:
        # Detach, ensure it's a float, but DO NOT move to CPU or convert to numpy
        W = m.weight.data.detach().float()
        W_list.append(W)
        layer_names.append(n)
        print(f"Matrix shape: {W.shape} {n}")

print(f"\nTotal layers extracted: {len(W_list)}")

Matrix shape: torch.Size([576, 576]) model.layers.0.self_attn.q_proj
Matrix shape: torch.Size([192, 576]) model.layers.0.self_attn.k_proj
Matrix shape: torch.Size([192, 576]) model.layers.0.self_attn.v_proj
Matrix shape: torch.Size([576, 576]) model.layers.0.self_attn.o_proj
Matrix shape: torch.Size([1536, 576]) model.layers.0.mlp.gate_proj
Matrix shape: torch.Size([1536, 576]) model.layers.0.mlp.up_proj
Matrix shape: torch.Size([576, 1536]) model.layers.0.mlp.down_proj
Matrix shape: torch.Size([576, 576]) model.layers.1.self_attn.q_proj
Matrix shape: torch.Size([192, 576]) model.layers.1.self_attn.k_proj
Matrix shape: torch.Size([192, 576]) model.layers.1.self_attn.v_proj
Matrix shape: torch.Size([576, 576]) model.layers.1.self_attn.o_proj
Matrix shape: torch.Size([1536, 576]) model.layers.1.mlp.gate_proj
Matrix shape: torch.Size([1536, 576]) model.layers.1.mlp.up_proj
Matrix shape: torch.Size([576, 1536]) model.layers.1.mlp.down_proj
Matrix shape: torch.Size([576, 576]) model.layers.

In [4]:
# Import our algorithms for pruning
from tetris import (
    tetris_pruning, 
    original_tetris_pruning, 
    random_swaps_find_mask, 
    sort_columns_by_norm, 
    block_sparsity_pruning
)

In [8]:
def benchmark_run(alg_name, pruning_func, *args, **kwargs):
    print(f"  Running {alg_name}...", end="", flush=True)
    
    W_tensor = args[0] 

    input_sq_norms = kwargs.pop('input_sq_norms', None)
    if input_sq_norms is None:
        input_sq_norms = torch.ones(W_tensor.shape[1], device=W_tensor.device)
    
    # Calculate baseline score for t=0 (Standard Pruning without permutation)
    _, mask_std = block_sparsity_pruning(W_tensor, kwargs.get('block_size', (1, 2)), kwargs.get('sparsity', 0.5))
    start_score = torch.abs(W_tensor)[mask_std == 0].sum().item()

    # Start precision timer
    start_time = time.perf_counter()
    
    # Execute the algorithm
    result = pruning_func(*args, **kwargs) 
    
    if W_tensor.is_cuda:
        torch.cuda.synchronize()
        
    total_time = time.perf_counter() - start_time
    print(f" Done! ({total_time:.4f}s)")
    
    # W2 is assumed to be the first returned object (the final un-permuted pruned matrix)
    W2 = result[0]
    
    # Calculate Relative Error
    input_sq_norms = kwargs.get('input_sq_norms', torch.ones(W_tensor.shape[1], device=W_tensor.device))
    rel_error = ((W_tensor - W2).float().square() * input_sq_norms).sum().item() / (W_tensor.float().square() * input_sq_norms).sum().item()
    
    # Extract results safely
    convergence_time = total_time
    if len(result) > 3:
        convergence_time = result[3]

    history = result[4] if len(result) > 4 else None
    after_tetris_index_in_history = result[5] if len(result) > 5 else None

    # If no history, create synthetic history [Start -> End]
    if history is None:
        final_score = torch.abs(W2)[result[1] == 0].sum().item()
        history = [(0.0, start_score), (total_time, final_score)]

    return {
        "Algorithm": alg_name,
        "Total Time": total_time,
        "Convergence Time": convergence_time,
        "Pruned Sum": history[-1][1],
        "Original Pruned Sum": history[0][1],
        "Improvement %": ((history[0][1] - history[-1][1]) / history[0][1] * 100) if history[0][1] > 0 else 0,
        "Relative Error": rel_error,
        "History": history,
        "Tetris Index": after_tetris_index_in_history
    }

In [11]:
# Global parameters
BLOCK_SIZE = (1, 2) 
SPARSITY = 0.5
MAX_ITER = 10
RANDOM_SWAPS = 10
NOISE_SCALE = 0.0

all_results = []

for i, (name, W) in enumerate(zip(layer_names, W_list)):
    print(f"\n--- Layer {i+1}/{len(W_list)}: {name} ---")
    
    dummy_input_norms = torch.ones(W.shape[1], device=W.device)
    
    # 1. Sort Columns
    res_sort = benchmark_run(
        "Sort Columns", 
        sort_columns_by_norm, 
        W, 
        block_size=BLOCK_SIZE, 
        sparsity=SPARSITY, 
        verbose=False,
        input_sq_norms=dummy_input_norms
    )
    res_sort["Layer"] = name
    all_results.append(res_sort)
    
    # 2. Original Tetris
    res_orig = benchmark_run(
        "Original Tetris", 
        original_tetris_pruning, 
        W, 
        block_size=BLOCK_SIZE, 
        sparsity=SPARSITY, 
        max_iter=MAX_ITER, 
        verbose=False,
        input_sq_norms=dummy_input_norms
    )
    res_orig["Layer"] = name
    all_results.append(res_orig)

    # 3. Random Swaps
    res_rand = benchmark_run(
        "Random Swaps", 
        random_swaps_find_mask, 
        W, 
        block_size=BLOCK_SIZE, 
        sparsity=SPARSITY,
        verbose=False,
        input_sq_norms=dummy_input_norms
    )
    res_rand["Layer"] = name
    all_results.append(res_rand)

    # 4. Our Tetris
    res_our = benchmark_run(
        "Our Tetris", 
        tetris_pruning, 
        W, 
        block_size=BLOCK_SIZE, 
        sparsity=SPARSITY, 
        max_iter=MAX_ITER, 
        random_swaps=RANDOM_SWAPS, 
        noise_scale=NOISE_SCALE, 
        verbose=False,
        input_sq_norms=dummy_input_norms
    )
    res_our["Layer"] = name
    all_results.append(res_our)


# print(stats_our_tetris)
# print(stats_random_swaps)
# print(stats_original_tetris)
# print(stats_sort_columns_by_norm)


--- Layer 1/210: model.layers.0.self_attn.q_proj ---
  Running Sort Columns...

 Done! (0.0013s)
  Running Original Tetris...Tetris iteration 10/10
 Done! (0.0530s)
  Running Random Swaps...Random swap iteration 10/10
 Done! (0.0345s)
  Running Our Tetris... Done! (0.2378s)

--- Layer 2/210: model.layers.0.self_attn.k_proj ---
  Running Sort Columns... Done! (0.0011s)
  Running Original Tetris...Tetris iteration 10/10
 Done! (0.0653s)
  Running Random Swaps...Random swap iteration 10/10
 Done! (0.0343s)
  Running Our Tetris... Done! (0.2448s)

--- Layer 3/210: model.layers.0.self_attn.v_proj ---
  Running Sort Columns... Done! (0.0011s)
  Running Original Tetris...Tetris iteration 10/10
 Done! (0.0495s)
  Running Random Swaps...Random swap iteration 10/10
 Done! (0.0354s)
  Running Our Tetris... Done! (0.2378s)

--- Layer 4/210: model.layers.0.self_attn.o_proj ---
  Running Sort Columns... Done! (0.0011s)
  Running Original Tetris...Tetris iteration 10/10
 Done! (0.0119s)
  Running Random Swaps...Random swap iteration 10/10
 Done! (0.0356s)
  Running Our Tetris...

In [12]:
from IPython.display import display

# Convert the results list to a Pandas DataFrame
df = pd.DataFrame(all_results)

# Create a macro summary by grouping by the Algorithm
summary_df = df.groupby("Algorithm").agg(
    Mean_Pruned_Sum=("Pruned Sum", "mean"),
    Mean_Improvement_Pct=("Improvement %", "mean"),
    Mean_Relative_Error=("Relative Error", "mean"),
    Mean_Total_Time_Sec=("Total Time", "mean")
).reset_index()

# Sort by the Pruned Sum (Lower is better)
summary_df = summary_df.sort_values("Mean_Pruned_Sum")

print("\n=== MACRO SUMMARY: AVERAGE PERFORMANCE ACROSS ALL LAYERS ===")
display(summary_df.round(4))


=== MACRO SUMMARY: AVERAGE PERFORMANCE ACROSS ALL LAYERS ===


,Algorithm,Mean_Pruned_Sum,Mean_Improvement_Pct,Mean_Relative_Error,Mean_Total_Time_Sec
2,Random Swaps,21006.9813,0.0271,1.9965,0.0416
3,Sort Columns,21011.6913,1.1125,1.9966,0.0011
1,Our Tetris,21134.8261,0.2659,1.0990,0.5211
0,Original Tetris,21155.0197,0.1081,0.0016,0.0477


In [13]:
# Save the full results to CSV
df.to_csv('tetris_benchmark_results.csv', index=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Pivot the DataFrame so Layers are rows and Algorithms are columns
pivot_df = df.pivot(index="Layer", columns="Algorithm", values="Pruned Sum")
pivot_df_err = df.pivot(index="Layer", columns="Algorithm", values="Relative Error")

# The layers might be strings, so we use integer indices for a clean x-axis
x_indices = np.arange(len(pivot_df))

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 15), sharex=True)

# --- TOP PLOT: Absolute Pruned Sum ---
for algo in pivot_df.columns:
    ax1.plot(x_indices, pivot_df[algo].values, label=algo, linewidth=1.5, alpha=0.85)

ax1.set_title('Total Sum of Pruned Weights Across Layers (Lower is Better)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Pruned Weight Sum', fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)

# --- MIDDLE PLOT: Difference from Baseline (Pruned Sum) ---
baseline_algo = "Original Tetris"
if baseline_algo not in pivot_df.columns:
    baseline_algo = pivot_df.columns[0]

baseline_values = pivot_df[baseline_algo].values

for algo in pivot_df.columns:
    if algo == baseline_algo:
        continue 
    
    # Calculate the delta (Algorithm Sum - Baseline Sum)
    delta = pivot_df[algo].values - baseline_values
    ax2.plot(x_indices, delta, label=algo, linewidth=1.5, alpha=0.85)

# Add the thick black line at 0 for the baseline
ax2.axhline(0, color='black', linestyle='-', linewidth=2, label=f'Baseline ({baseline_algo})')

ax2.set_title(f'Difference in Pruned Sum (Compared to {baseline_algo})', fontsize=14, fontweight='bold')
ax2.set_ylabel('Difference\n(Negative is Better)', fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)

# --- BOTTOM PLOT: Relative Error ---
for algo in pivot_df_err.columns:
    ax3.plot(x_indices, pivot_df_err[algo].values, label=algo, linewidth=1.5, alpha=0.85)

ax3.set_title('Relative Output Activation Error (Lower is Better)', fontsize=14, fontweight='bold')
ax3.set_xlabel('Layer Index', fontsize=12)
ax3.set_ylabel('Relative Error', fontsize=12)
ax3.grid(True, linestyle='--', alpha=0.6)
ax3.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined